# Chatbot 
- Seq2Seq Model Q&A Chatbot 
<img src='https://media.geeksforgeeks.org/wp-content/uploads/20250529130320642115/Seq2Seq-Model.webp'>
1. find QnA dataset & preprocessing
2. seq to seq : Encoder, Decoder, Seq2Seq 모델 만든다
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습시킨다 
3. Chatbot을 만든다 (모델 추론 + while문) 
- objective: repetitions of input & 적절한 output
while: 
    input -> encoder - decoder -> output 
- https://www.kaggle.com/datasets/taejinwoo/multiwoz-22/data

**Questions**
- lemmatization? 

In [11]:
# import pandas as pd

# df = pd.read_csv('https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv')
# df = df[['Q', 'A']]
# df

In [12]:
# Download Kaggle Librarty to use Kaggle Data
# !pip install -qqq kagglehub pandas numpy tensorflow

In [13]:
import kagglehub
import os
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Download latest version
data_path = kagglehub.dataset_download("taejinwoo/multiwoz-22")
print("Path to dataset files:", data_path)

def load_multiwoz_pairs(data_path, max_pairs=None):
    pairs = []
    for root, dirs, files in os.walk(data_path):
        for filename in files:
            if filename.endswith(".json") and "dialogues" in filename:
                full_path = os.path.join(root, filename)
                with open(full_path, "r") as f:
                    data = json.load(f)
                    for dialog in data:
                        turns = dialog["turns"]
                        for i in range(0, len(turns) - 1, 2):
                            user = turns[i]["utterance"]
                            system = turns[i + 1]["utterance"]
                            pairs.append((user, system))
                            if max_pairs and len(pairs) >= max_pairs:
                                return pairs
    return pairs

pairs = load_multiwoz_pairs(data_path, max_pairs=30000)  # limit for faster training
df = pd.DataFrame(pairs, columns=["Q", "A"])
df.head()


Path to dataset files: /Users/kat/.cache/kagglehub/datasets/taejinwoo/multiwoz-22/versions/1


,Q,A
0,I need train reservations from norwich to camb...,I have 133 trains matching your request. Is th...
1,I'd like to leave on Monday and arrive by 18:00.,There are 12 trains for the day and time you r...
2,"Before booking, I would also like to know the ...",There are 12 trains meeting your needs with th...
3,No hold off on booking for now. Can you help m...,Yes it is a cinema located in the south part o...
4,"Yes, that was all I needed. Thank you very much!",Thank you for using our system.


### preprocessing

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import nltk

nltk.download("wordnet")
nltk.download("punkt")

lemmatizer = WordNetLemmatizer()
en_stopwords = set(stopwords.words("english"))

def clean_sentence(sentence):
    sentence = re.sub(r"[^\x00-\x7F]+", " ", sentence)  # remove non ascii 
    sentence = re.sub(r"[^a-zA-Z0-9\s?.!,]", "", sentence.lower())
    sentence = sentence.strip()
    sentence = re.sub(r"\s+", " ", sentence)    # multiple whitespaces -> one
    return sentence

def tokenize_lemmatize(sentence):
    tokens = word_tokenize(sentence)
    lemmatized = [lemmatizer.lemmatize(tok, pos="v") for tok in tokens]
    return lemmatized

def preprocess_text(text):
    cleaned = clean_sentence(text)
    tokens = tokenize_lemmatize(cleaned)
    return tokens

df["Q_prep"] = df["Q"].apply(preprocess_text)
df["A_prep"] = df["A"].apply(preprocess_text)

df[["Q", "Q_prep", "A", "A_prep"]].head()


[nltk_data] Downloading package wordnet to /Users/kat/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /Users/kat/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,Q,Q_prep,A,A_prep
0,i need train reservations from norwich to camb...,"[i, need, train, reservations, from, norwich, ...",i have 133 trains matching your request. is th...,"[i, have, 133, train, match, your, request, .,..."
1,i'd like to leave on monday and arrive by 18:00.,"[id, like, to, leave, on, monday, and, arrive,...",there are 12 trains for the day and time you r...,"[there, be, 12, train, for, the, day, and, tim..."
2,"before booking, i would also like to know the ...","[before, book, ,, i, would, also, like, to, kn...",there are 12 trains meeting your needs with th...,"[there, be, 12, train, meet, your, need, with,..."
3,no hold off on booking for now. can you help m...,"[no, hold, off, on, book, for, now, ., can, yo...",yes it is a cinema located in the south part o...,"[yes, it, be, a, cinema, locate, in, the, sout..."
4,"yes, that was all i needed. thank you very much!","[yes, ,, that, be, all, i, need, ., thank, you...",thank you for using our system.,"[thank, you, for, use, our, system, .]"


[nltk_data] Downloading package wordnet to /Users/kat/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /Users/kat/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,Q,Q_prep,A,A_prep
0,i need train reservations from norwich to camb...,"[i, need, train, reservations, from, norwich, ...",i have 133 trains matching your request. is th...,"[i, have, 133, train, match, your, request, .,..."
1,i'd like to leave on monday and arrive by 18:00.,"[id, like, to, leave, on, monday, and, arrive,...",there are 12 trains for the day and time you r...,"[there, be, 12, train, for, the, day, and, tim..."
2,"before booking, i would also like to know the ...","[before, book, ,, i, would, also, like, to, kn...",there are 12 trains meeting your needs with th...,"[there, be, 12, train, meet, your, need, with,..."
3,no hold off on booking for now. can you help m...,"[no, hold, off, on, book, for, now, ., can, yo...",yes it is a cinema located in the south part o...,"[yes, it, be, a, cinema, locate, in, the, sout..."
4,"yes, that was all i needed. thank you very much!","[yes, ,, that, be, all, i, need, ., thank, you...",thank you for using our system.,"[thank, you, for, use, our, system, .]"


### 데이터 학습 준비 

In [ ]:
question_inputs = []
answer_inputs = []
answer_targets = []

for q_tokens, a_tokens in zip(df["Q_prep"], df["A_prep"]):
    question_inputs.append(q_tokens)
    answer_inputs.append(["<sos>"] + a_tokens)  # start of sentence
    answer_targets.append(a_tokens + ["<eos>"]) # end of sentence answer (ground truth)

print(len(question_inputs), len(answer_inputs), len(answer_targets))
print(question_inputs[0])
print(answer_inputs[0])
print(answer_targets[0])


30000 30000 30000
['i', 'need', 'train', 'reservations', 'from', 'norwich', 'to', 'cambridge']
['<sos>', 'i', 'have', '133', 'train', 'match', 'your', 'request', '.', 'be', 'there', 'a', 'specific', 'day', 'and', 'time', 'you', 'would', 'like', 'to', 'travel', '?']
['i', 'have', '133', 'train', 'match', 'your', 'request', '.', 'be', 'there', 'a', 'specific', 'day', 'and', 'time', 'you', 'would', 'like', 'to', 'travel', '?', '<eos>']


### 전역변수 선언
- 배치 크기, 단어사전 크기, 임베딩 차원, 뉴런수

In [16]:
BATCH_SIZE = 64
MAX_VOCAB_SIZE = 10000
EMBEDDING_DIM = 100
LATENT_DIM = 256  # reduce from 512 for speed
MAX_LEN = 30      # truncate for speed

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


question_texts = [" ".join(tokens[:MAX_LEN]) for tokens in question_inputs]
answer_input_texts = [" ".join(tokens[:MAX_LEN]) for tokens in answer_inputs]
answer_target_texts = [" ".join(tokens[:MAX_LEN]) for tokens in answer_targets]

# vocab & vectorization
question_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<unk>")
question_tokenizer.fit_on_texts(question_texts) 
question_seqs = question_tokenizer.texts_to_sequences(question_texts)

answer_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters="", oov_token="<unk>")
answer_tokenizer.fit_on_texts(answer_input_texts + answer_target_texts) # decoder library
answer_input_seqs = answer_tokenizer.texts_to_sequences(answer_input_texts)
answer_target_seqs = answer_tokenizer.texts_to_sequences(answer_target_texts)

question_num = len(question_tokenizer.word_index) + 1   # num of unique words + 1 for padding
answer_num = len(answer_tokenizer.word_index) + 1

question_max_len = min(MAX_LEN, max(len(s) for s in question_seqs))
answer_max_len = min(MAX_LEN, max(len(s) for s in answer_input_seqs))

print(f"question_num={question_num}, question_max_len={question_max_len}")
print(f"answer_num={answer_num}, answer_max_len={answer_max_len}")


question_num=2773, question_max_len=30
answer_num=9813, answer_max_len=30


In [ ]:
# encoder_inputs = pad_sequences(question_seqs, maxlen=question_max_len, padding="post", truncating="post")
# decoder_inputs = pad_sequences(answer_input_seqs, maxlen=answer_max_len, padding="post", truncating="post")
# decoder_targets = pad_sequences(answer_target_seqs, maxlen=answer_max_len, padding="post", truncating="post")

# print("encoder_inputs.shape:", encoder_inputs.shape)
# print("decoder_inputs.shape:", decoder_inputs.shape)
# print("decoder_targets.shape:", decoder_targets.shape)


encoder_inputs.shape: (30000, 30)
decoder_inputs.shape: (30000, 30)
decoder_targets.shape: (30000, 30)


In [ ]:
# padding post cuz lstm (process real words first not padding)
encoder_inputs = pad_sequences(question_seqs, maxlen=question_max_len, padding='post') # idk why post
decoder_inputs = pad_sequences(answer_input_seqs, maxlen=answer_max_len, padding='post')
decoder_targets = pad_sequences(answer_target_seqs, maxlen=answer_max_len, padding='post')

print("encoder_inputs.shape: ", encoder_inputs.shape)
print("decoder_inputs.shape: ", decoder_inputs.shape)
print("decoder_targets.shape: ", decoder_targets.shape)

encoder_inputs.shape:  (30000, 30)
decoder_inputs.shape:  (30000, 30)
decoder_targets.shape:  (30000, 30)


In [20]:
class NMTDataset(Dataset):  # Multi-Modal Transformer Dataset
  def __init__(self, encoder_inputs, decoder_inputs, decoder_targets):
    super().__init__()
    self.encoder_inputs = encoder_inputs
    self.decoder_inputs = decoder_inputs
    self.decoder_targets = decoder_targets

  def __len__(self):
    return len(self.encoder_inputs)

  def __getitem__(self, index):
    return (
        torch.tensor(self.encoder_inputs[index], dtype=torch.long),
        torch.tensor(self.decoder_inputs[index], dtype=torch.long),
        torch.tensor(self.decoder_targets[index], dtype=torch.long),
    )

class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim, embedding_matrix=None):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    if embedding_matrix is not None:
      self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)

  def forward(self, X):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X)
    return h_s, c_s

class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)
    self.fc = nn.Linear(latent_dim, vocab_size)

  def forward(self, X, hidden, cell):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X, (hidden, cell))
    logits = self.fc(output)
    return logits, h_s, c_s




# Seq2Seq Model
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, source, target):
    h_s, c_s = self.encoder(source)
    output, h_s, c_s = self.decoder(target, h_s, c_s)
    return output


In [ ]:
train_index, val_index = train_test_split(range(len(encoder_inputs)), random_state=0)
print(len(train_index), len(val_index))

train_dataset = NMTDataset(
    encoder_inputs[train_index],
    decoder_inputs[train_index],
    decoder_targets[train_index]
)
val_dataset = NMTDataset(
    encoder_inputs[val_index],
    decoder_inputs[val_index],
    decoder_targets[val_index]
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

22500 7500


### 모델 준비
- GloVe (pretrained)
- word embedding: to understand relationships between words (semantic nuance)

In [ ]:
import numpy as np
import os

def make_embedding_matrix(num_words, embedding_dim, tokenizer, cache_path="embedding_matrix.npy"):
    if os.path.exists(cache_path):
        print("Loading cached embedding matrix...")
        return np.load(cache_path)

    embedding_dict = {}
    with open("06_data/glove.6B.100d.txt", "r", encoding="utf-8") as f:
        for line in f:
            word, *coef = line.split()
            embedding_dict[word] = np.array(coef, dtype=float)

    embedding_matrix = np.zeros((num_words, embedding_dim))
    for word, index in tokenizer.word_index.items():
        if index >= num_words:
            continue
        vec = embedding_dict.get(word)
        if vec is not None:
            embedding_matrix[index] = vec
        else:
            embedding_matrix[index] = np.random.rand(embedding_dim)

    np.save(cache_path, embedding_matrix)
    return embedding_matrix

embedding_matrix = make_embedding_matrix(question_num, EMBEDDING_DIM, question_tokenizer)
print(embedding_matrix.shape)


Loading cached embedding matrix...
(2773, 100)


In [23]:
encoder = Encoder(question_num, EMBEDDING_DIM, LATENT_DIM, embedding_matrix)
decoder = Decoder(answer_num, EMBEDDING_DIM, LATENT_DIM)

model = Seq2Seq(encoder, decoder)

In [24]:
model

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(2773, 100, padding_idx=0)
    (lstm): LSTM(100, 256, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(9813, 100, padding_idx=0)
    (lstm): LSTM(100, 256, batch_first=True)
    (fc): Linear(in_features=256, out_features=9813, bias=True)
  )
)

### 모델 학습

In [ ]:
# seq2seq teacher forcing training  

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(question_num, EMBEDDING_DIM, LATENT_DIM, embedding_matrix)
decoder = Decoder(answer_num, EMBEDDING_DIM, LATENT_DIM)
model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

epochs = 100 #todo edit to 100 

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):
  model.train()
  train_loss, train_correct, train_tokens = 0, 0, 0

  for enc_inputs, dec_inputs, dec_targets in train_loader:
    enc_inputs = enc_inputs.to(device)
    dec_inputs = dec_inputs.to(device)
    dec_targets = dec_targets.to(device)

    optimizer.zero_grad()

    # teacher forcing
    output = model(enc_inputs, dec_inputs)  # answer_inputs dec_inputs gives answer to decode
    output = output.view(-1, output.size(-1))
    dec_targets = dec_targets.view(-1)

    loss = criterion(output, dec_targets)
    loss.backward()
    optimizer.step()

    preds = output.argmax(dim=-1)
    train_loss += loss.detach().cpu().item()
    mask = dec_targets != 0
    correct = (preds == dec_targets) & mask
    train_correct += correct.sum().detach().cpu().item()
    train_tokens += mask.sum().detach().cpu().item()

  train_loss /= len(train_loader)
  train_acc = train_correct / train_tokens
  train_losses.append(train_loss)
  train_accs.append(train_acc)

  model.eval()
  with torch.no_grad():
    val_loss, val_correct, val_tokens = 0, 0, 0

    for enc_inputs, dec_inputs, dec_targets in val_loader:
      enc_inputs = enc_inputs.to(device)
      dec_inputs = dec_inputs.to(device)
      dec_targets = dec_targets.to(device)

      output = model(enc_inputs, dec_inputs)
      output = output.view(-1, output.size(-1))
      dec_targets = dec_targets.view(-1)

      loss = criterion(output, dec_targets)

      preds = output.argmax(dim=-1)
      val_loss += loss.detach().cpu().item()
      mask = dec_targets != 0
      correct = (preds == dec_targets) & mask
      val_correct += correct.sum().detach().cpu().item()
      val_tokens += mask.sum().detach().cpu().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens
    val_losses.append(val_loss)
    val_accs.append(val_acc)

  print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

Epoch 1/100 TrainLoss=4.2636 TrainAcc=0.2740 ValLoss=3.3031 ValAcc=0.3851
Epoch 2/100 TrainLoss=2.9592 TrainAcc=0.4214 ValLoss=2.7893 ValAcc=0.4463
Epoch 3/100 TrainLoss=2.5596 TrainAcc=0.4702 ValLoss=2.5361 ValAcc=0.4785
Epoch 4/100 TrainLoss=2.3343 TrainAcc=0.4989 ValLoss=2.3994 ValAcc=0.4982
Epoch 5/100 TrainLoss=2.1913 TrainAcc=0.5157 ValLoss=2.3146 ValAcc=0.5100
Epoch 6/100 TrainLoss=2.0846 TrainAcc=0.5290 ValLoss=2.2565 ValAcc=0.5195
Epoch 7/100 TrainLoss=2.0017 TrainAcc=0.5393 ValLoss=2.2126 ValAcc=0.5260
Epoch 8/100 TrainLoss=1.9297 TrainAcc=0.5484 ValLoss=2.1817 ValAcc=0.5301
Epoch 9/100 TrainLoss=1.8660 TrainAcc=0.5569 ValLoss=2.1587 ValAcc=0.5354
Epoch 10/100 TrainLoss=1.8116 TrainAcc=0.5645 ValLoss=2.1404 ValAcc=0.5390
Epoch 11/100 TrainLoss=1.7609 TrainAcc=0.5717 ValLoss=2.1254 ValAcc=0.5420
Epoch 12/100 TrainLoss=1.7133 TrainAcc=0.5791 ValLoss=2.1231 ValAcc=0.5425
Epoch 13/100 TrainLoss=1.6691 TrainAcc=0.5862 ValLoss=2.1145 ValAcc=0.5449
Epoch 14/100 TrainLoss=1.6270 Trai

In [ ]:
# Epoch 1/100 TrainLoss=4.2636 TrainAcc=0.2740 ValLoss=3.3031 ValAcc=0.3851
# Epoch 2/100 TrainLoss=2.9592 TrainAcc=0.4214 ValLoss=2.7893 ValAcc=0.4463
# Epoch 3/100 TrainLoss=2.5596 TrainAcc=0.4702 ValLoss=2.5361 ValAcc=0.4785
# Epoch 4/100 TrainLoss=2.3343 TrainAcc=0.4989 ValLoss=2.3994 ValAcc=0.4982
# Epoch 5/100 TrainLoss=2.1913 TrainAcc=0.5157 ValLoss=2.3146 ValAcc=0.5100
# Epoch 6/100 TrainLoss=2.0846 TrainAcc=0.5290 ValLoss=2.2565 ValAcc=0.5195
# Epoch 7/100 TrainLoss=2.0017 TrainAcc=0.5393 ValLoss=2.2126 ValAcc=0.5260
# Epoch 8/100 TrainLoss=1.9297 TrainAcc=0.5484 ValLoss=2.1817 ValAcc=0.5301
# Epoch 9/100 TrainLoss=1.8660 TrainAcc=0.5569 ValLoss=2.1587 ValAcc=0.5354
# Epoch 10/100 TrainLoss=1.8116 TrainAcc=0.5645 ValLoss=2.1404 ValAcc=0.5390
# Epoch 11/100 TrainLoss=1.7609 TrainAcc=0.5717 ValLoss=2.1254 ValAcc=0.5420
# Epoch 12/100 TrainLoss=1.7133 TrainAcc=0.5791 ValLoss=2.1231 ValAcc=0.5425
# Epoch 13/100 TrainLoss=1.6691 TrainAcc=0.5862 ValLoss=2.1145 ValAcc=0.5449
# Epoch 14/100 TrainLoss=1.6270 TrainAcc=0.5930 ValLoss=2.1140 ValAcc=0.5465
# Epoch 15/100 TrainLoss=1.5875 TrainAcc=0.5997 ValLoss=2.1158 ValAcc=0.5471
# Epoch 16/100 TrainLoss=1.5490 TrainAcc=0.6067 ValLoss=2.1181 ValAcc=0.5466
# Epoch 17/100 TrainLoss=1.5121 TrainAcc=0.6140 ValLoss=2.1223 ValAcc=0.5472
# Epoch 18/100 TrainLoss=1.4767 TrainAcc=0.6208 ValLoss=2.1286 ValAcc=0.5489
# Epoch 19/100 TrainLoss=1.4422 TrainAcc=0.6276 ValLoss=2.1320 ValAcc=0.5474
# Epoch 20/100 TrainLoss=1.4080 TrainAcc=0.6348 ValLoss=2.1388 ValAcc=0.5478
# Epoch 21/100 TrainLoss=1.3754 TrainAcc=0.6420 ValLoss=2.1502 ValAcc=0.5463
# Epoch 22/100 TrainLoss=1.3440 TrainAcc=0.6489 ValLoss=2.1617 ValAcc=0.5472
# Epoch 23/100 TrainLoss=1.3130 TrainAcc=0.6553 ValLoss=2.1723 ValAcc=0.5463
# Epoch 24/100 TrainLoss=1.2839 TrainAcc=0.6623 ValLoss=2.1886 ValAcc=0.5457
# Epoch 25/100 TrainLoss=1.2546 TrainAcc=0.6689 ValLoss=2.2012 ValAcc=0.5438
# Epoch 26/100 TrainLoss=1.2261 TrainAcc=0.6754 ValLoss=2.2139 ValAcc=0.5443
# Epoch 27/100 TrainLoss=1.1978 TrainAcc=0.6821 ValLoss=2.2318 ValAcc=0.5433
# Epoch 28/100 TrainLoss=1.1723 TrainAcc=0.6878 ValLoss=2.2450 ValAcc=0.5421
# Epoch 29/100 TrainLoss=1.1450 TrainAcc=0.6943 ValLoss=2.2630 ValAcc=0.5414
# Epoch 30/100 TrainLoss=1.1208 TrainAcc=0.7005 ValLoss=2.2769 ValAcc=0.5412
# Epoch 31/100 TrainLoss=1.0954 TrainAcc=0.7068 ValLoss=2.2936 ValAcc=0.5404
# Epoch 32/100 TrainLoss=1.0717 TrainAcc=0.7128 ValLoss=2.3080 ValAcc=0.5392
# Epoch 33/100 TrainLoss=1.0466 TrainAcc=0.7191 ValLoss=2.3375 ValAcc=0.5372
# Epoch 34/100 TrainLoss=1.0246 TrainAcc=0.7246 ValLoss=2.3480 ValAcc=0.5356
# Epoch 35/100 TrainLoss=1.0016 TrainAcc=0.7304 ValLoss=2.3696 ValAcc=0.5342
# Epoch 36/100 TrainLoss=0.9790 TrainAcc=0.7363 ValLoss=2.3889 ValAcc=0.5342
# Epoch 37/100 TrainLoss=0.9581 TrainAcc=0.7419 ValLoss=2.4102 ValAcc=0.5342
# Epoch 38/100 TrainLoss=0.9381 TrainAcc=0.7470 ValLoss=2.4332 ValAcc=0.5306
# Epoch 39/100 TrainLoss=0.9173 TrainAcc=0.7520 ValLoss=2.4486 ValAcc=0.5314
# Epoch 40/100 TrainLoss=0.8973 TrainAcc=0.7575 ValLoss=2.4722 ValAcc=0.5295
# Epoch 41/100 TrainLoss=0.8771 TrainAcc=0.7633 ValLoss=2.4913 ValAcc=0.5281
# Epoch 42/100 TrainLoss=0.8581 TrainAcc=0.7682 ValLoss=2.5053 ValAcc=0.5275
# Epoch 43/100 TrainLoss=0.8397 TrainAcc=0.7729 ValLoss=2.5336 ValAcc=0.5256
# Epoch 44/100 TrainLoss=0.8223 TrainAcc=0.7777 ValLoss=2.5557 ValAcc=0.5247
# Epoch 45/100 TrainLoss=0.8046 TrainAcc=0.7827 ValLoss=2.5752 ValAcc=0.5225
# Epoch 46/100 TrainLoss=0.7869 TrainAcc=0.7875 ValLoss=2.5966 ValAcc=0.5198
# Epoch 47/100 TrainLoss=0.7697 TrainAcc=0.7916 ValLoss=2.6157 ValAcc=0.5196
# Epoch 48/100 TrainLoss=0.7555 TrainAcc=0.7953 ValLoss=2.6404 ValAcc=0.5195
# Epoch 49/100 TrainLoss=0.7395 TrainAcc=0.8005 ValLoss=2.6615 ValAcc=0.5194
# Epoch 50/100 TrainLoss=0.7221 TrainAcc=0.8045 ValLoss=2.6885 ValAcc=0.5168
# Epoch 51/100 TrainLoss=0.7077 TrainAcc=0.8091 ValLoss=2.7072 ValAcc=0.5172
# Epoch 52/100 TrainLoss=0.6934 TrainAcc=0.8130 ValLoss=2.7279 ValAcc=0.5151
# Epoch 53/100 TrainLoss=0.6783 TrainAcc=0.8170 ValLoss=2.7459 ValAcc=0.5150
# Epoch 54/100 TrainLoss=0.6641 TrainAcc=0.8210 ValLoss=2.7681 ValAcc=0.5132
# Epoch 55/100 TrainLoss=0.6517 TrainAcc=0.8247 ValLoss=2.7879 ValAcc=0.5122
# Epoch 56/100 TrainLoss=0.6392 TrainAcc=0.8278 ValLoss=2.8164 ValAcc=0.5113
# Epoch 57/100 TrainLoss=0.6263 TrainAcc=0.8314 ValLoss=2.8367 ValAcc=0.5109
# Epoch 58/100 TrainLoss=0.6126 TrainAcc=0.8354 ValLoss=2.8533 ValAcc=0.5096
# Epoch 59/100 TrainLoss=0.6016 TrainAcc=0.8389 ValLoss=2.8826 ValAcc=0.5084
# Epoch 60/100 TrainLoss=0.5876 TrainAcc=0.8424 ValLoss=2.9054 ValAcc=0.5072
# Epoch 61/100 TrainLoss=0.5760 TrainAcc=0.8453 ValLoss=2.9272 ValAcc=0.5073
# Epoch 62/100 TrainLoss=0.5664 TrainAcc=0.8481 ValLoss=2.9480 ValAcc=0.5052
# Epoch 63/100 TrainLoss=0.5534 TrainAcc=0.8525 ValLoss=2.9714 ValAcc=0.5057
# Epoch 64/100 TrainLoss=0.5404 TrainAcc=0.8563 ValLoss=2.9857 ValAcc=0.5044
# Epoch 65/100 TrainLoss=0.5314 TrainAcc=0.8584 ValLoss=3.0130 ValAcc=0.5059
# Epoch 66/100 TrainLoss=0.5225 TrainAcc=0.8607 ValLoss=3.0260 ValAcc=0.5032
# Epoch 67/100 TrainLoss=0.5148 TrainAcc=0.8630 ValLoss=3.0491 ValAcc=0.5020
# Epoch 68/100 TrainLoss=0.5030 TrainAcc=0.8666 ValLoss=3.0751 ValAcc=0.5032
# Epoch 69/100 TrainLoss=0.4940 TrainAcc=0.8690 ValLoss=3.0914 ValAcc=0.5018
# Epoch 70/100 TrainLoss=0.4847 TrainAcc=0.8717 ValLoss=3.1157 ValAcc=0.5005
# Epoch 71/100 TrainLoss=0.4743 TrainAcc=0.8750 ValLoss=3.1454 ValAcc=0.5002
# Epoch 72/100 TrainLoss=0.4649 TrainAcc=0.8779 ValLoss=3.1605 ValAcc=0.5000
# Epoch 73/100 TrainLoss=0.4568 TrainAcc=0.8798 ValLoss=3.1696 ValAcc=0.5009
# Epoch 74/100 TrainLoss=0.4505 TrainAcc=0.8817 ValLoss=3.1901 ValAcc=0.4998
# Epoch 75/100 TrainLoss=0.4438 TrainAcc=0.8833 ValLoss=3.2091 ValAcc=0.4980
# Epoch 76/100 TrainLoss=0.4318 TrainAcc=0.8868 ValLoss=3.2354 ValAcc=0.4976
# Epoch 77/100 TrainLoss=0.4260 TrainAcc=0.8891 ValLoss=3.2608 ValAcc=0.4962
# Epoch 78/100 TrainLoss=0.4166 TrainAcc=0.8916 ValLoss=3.2733 ValAcc=0.4959
# Epoch 79/100 TrainLoss=0.4095 TrainAcc=0.8935 ValLoss=3.2925 ValAcc=0.4960
# Epoch 80/100 TrainLoss=0.4007 TrainAcc=0.8960 ValLoss=3.3081 ValAcc=0.4950
# Epoch 81/100 TrainLoss=0.3946 TrainAcc=0.8979 ValLoss=3.3319 ValAcc=0.4953
# Epoch 82/100 TrainLoss=0.3888 TrainAcc=0.8995 ValLoss=3.3560 ValAcc=0.4942
# Epoch 83/100 TrainLoss=0.3893 TrainAcc=0.8987 ValLoss=3.3661 ValAcc=0.4941
# Epoch 84/100 TrainLoss=0.3773 TrainAcc=0.9023 ValLoss=3.3944 ValAcc=0.4924
# Epoch 85/100 TrainLoss=0.3648 TrainAcc=0.9064 ValLoss=3.4046 ValAcc=0.4921
# Epoch 86/100 TrainLoss=0.3619 TrainAcc=0.9075 ValLoss=3.4188 ValAcc=0.4937
# Epoch 87/100 TrainLoss=0.3586 TrainAcc=0.9073 ValLoss=3.4500 ValAcc=0.4913
# Epoch 88/100 TrainLoss=0.3573 TrainAcc=0.9080 ValLoss=3.4669 ValAcc=0.4903
# Epoch 89/100 TrainLoss=0.3465 TrainAcc=0.9114 ValLoss=3.4787 ValAcc=0.4915
# Epoch 90/100 TrainLoss=0.3383 TrainAcc=0.9137 ValLoss=3.5069 ValAcc=0.4918
# Epoch 91/100 TrainLoss=0.3331 TrainAcc=0.9154 ValLoss=3.5084 ValAcc=0.4911
# Epoch 92/100 TrainLoss=0.3309 TrainAcc=0.9156 ValLoss=3.5294 ValAcc=0.4916
# Epoch 93/100 TrainLoss=0.3226 TrainAcc=0.9182 ValLoss=3.5580 ValAcc=0.4894
# Epoch 94/100 TrainLoss=0.3164 TrainAcc=0.9204 ValLoss=3.5713 ValAcc=0.4895
# Epoch 95/100 TrainLoss=0.3126 TrainAcc=0.9211 ValLoss=3.5862 ValAcc=0.4910
# Epoch 96/100 TrainLoss=0.3111 TrainAcc=0.9210 ValLoss=3.6039 ValAcc=0.4885
# Epoch 97/100 TrainLoss=0.3060 TrainAcc=0.9225 ValLoss=3.6204 ValAcc=0.4897
# Epoch 98/100 TrainLoss=0.2997 TrainAcc=0.9245 ValLoss=3.6410 ValAcc=0.4894
# Epoch 99/100 TrainLoss=0.2964 TrainAcc=0.9259 ValLoss=3.6569 ValAcc=0.4890
# Epoch 100/100 TrainLoss=0.2944 TrainAcc=0.9258 ValLoss=3.6736 ValAcc=0.4880

In [ ]:
#Can I book a room?
#How to get to the train station?
#How much is coffee?
# How much is this ticket?
# Where is the best park?

### predicting
- Greedy Decoding: most probable word selected

In [ ]:
def chat(input_str, model, encoder, decoder, q_tokenizer, a_tokenizer, max_len=30):
    # preprocess
    tokens = preprocess_text(input_str)
    seq = q_tokenizer.texts_to_sequences([" ".join(tokens)])
    padded_seq = pad_sequences(seq, maxlen=question_max_len, padding='post')
    
    model.eval()

   
    with torch.no_grad():
        source = torch.tensor(padded_seq, dtype=torch.long) #input
        h_s, c_s = encoder(source)  # lstm encoding  
        
        # output start with start of sentence(sos)
        decoder_input = torch.tensor([[a_tokenizer.word_index['<sos>']]], dtype=torch.long)
        
        predicted_sentence = []

       
        for _ in range(max_len):     # autoregressive
            logits, h_s, c_s = decoder(decoder_input, h_s, c_s)
            
            top_token_idx = torch.argmax(logits[0, 0], dim=0).item()    #greedy search
            
            if top_token_idx == a_tokenizer.word_index.get('<eos>', -1):    #if <eos> stop
                break
                
            predicted_word = a_tokenizer.index_word.get(top_token_idx, '')  #index->word
            if predicted_word:
                predicted_sentence.append(predicted_word)   # add word to sentence
            
            decoder_input = torch.tensor([[top_token_idx]], dtype=torch.long)   # output as input
            
    return " ".join(predicted_sentence)

def start_chat():
    print("Ask me anything! Type 'q' to exit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'q':
            break
        
        response = chat(user_input, model, encoder, decoder, 
                                question_tokenizer, answer_tokenizer)
        print(f"Chatbot: {response}")

start_chat()

Chatbot initialized. Type 'quit' to exit.
